# 04. Evaluation harness + leakage study

**Lock down how we measure before retraining anything.**

Uses `src/evaluate.py`, which is reused by every model and seed from here on.
Builds three test sets with the project's `clean_split`/`remove_leakage`, so counts reconcile exactly:
- **clean:** leakage removed (your published test set, 41,957 rows)
- **leaky:** clean + test rows that also appear in train (the paper's condition)
- **leaked-only:** just the overlapping rows, isolating memorization

To evaluate a different model later, change only `MODEL_REPO` / `MAX_LENGTH` and re-run.


In [ ]:
# CONFIG
MODEL_REPO      = "iamahmadyasin/humor-intelligence-distilbert"
MAX_LENGTH      = 128
BATCH_SIZE      = 64
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/humor-intelligence.git"

!pip install -q "transformers>=4.40,<4.51" "datasets>=2.19"

import os, sys
if not os.path.exists("humor-intelligence"):
    !git clone -q $GITHUB_REPO_URL
REPO_DIR = os.path.abspath("humor-intelligence")
!cd "$REPO_DIR" && python src/data.py
sys.path.append(os.path.join(REPO_DIR, "src"))
print("repo:", REPO_DIR)

In [ ]:
# 1. Load model + tokenizer
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_REPO)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval().to(device)
print("Loaded", MODEL_REPO, "on", device)

In [ ]:
# 2. Build clean / leaky / leaked-only
import pandas as pd
from evaluate import build_test_variants, predict, compute_metrics

variants = build_test_variants(min_words=5)
clean, leaky, leaked = variants["clean"], variants["leaky"], variants["leaked_only"]

print(f"clean test : {len(clean):>7,}   (expected 41,957)")
print(f"leaked only: {len(leaked):>7,}   (expected ~1,000; ~2.4% of test)")
print(f"leaky test : {len(leaky):>7,}   (clean + leaked, asserted in build_test_variants)")

In [ ]:
# 3. Evaluate the SAME model on all three sets
rows = []
for name, df in [("clean", clean), ("leaky", leaky), ("leaked-only", leaked)]:
    pred = predict(model, tokenizer, df["joke"].tolist(), max_length=MAX_LENGTH, batch_size=BATCH_SIZE)
    rows.append({"eval_set": name, **compute_metrics(df["score"].values, pred)})

report = (pd.DataFrame(rows).set_index("eval_set")[["n", "spearman", "pearson", "rmse", "mae"]]
          .round({"spearman": 4, "pearson": 4, "rmse": 4, "mae": 4}))
print(report)
report